# 02 Model: Predicting dropout
Research questions 2 and 3: How early can we tell who will drop out, and which model works best?

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score, confusion_matrix

df = pd.read_csv('../data/data.csv', sep=';')
df.columns = df.columns.str.strip()
df['dropout'] = (df['Target'] == 'Dropout').astype(int)

**Notes:** Loaded the same 4,424 students and the dropout column (1 = dropped out, 0 = graduated or still enrolled). Imported scikit-learn tools for splitting data, preparing features, three models, and scoring.

In [24]:
CATEGORICAL = ['Marital status', 'Application mode', 'Course', 'Previous qualification',
               'Nacionality', "Mother's qualification", "Father's qualification",
               "Mother's occupation", "Father's occupation"]
SEM1 = [c for c in df.columns if c.startswith('Curricular units 1st sem')]
SEM2 = [c for c in df.columns if c.startswith('Curricular units 2nd sem')]
DROP = ['Target', 'dropout'] + SEM2

enroll_features = [c for c in df.columns if c not in DROP + SEM1]
sem1_features = enroll_features + SEM1
print(len(enroll_features), len(sem1_features))

24 30


**Notes:** I built two feature sets to answer "how early can we tell?"
- **At enrollment (24 features):** what the college knows on day 1: background, family, finances, admission grade, and the economy. No grades.
- **After semester 1 (30 features):** the same, plus 6 semester 1 grade columns.

I left out all semester 2 columns because an early warning tool has to act before semester 2. Nine columns are codes (like Course or parents' education), so the model treats them as categories, not amounts.

In [25]:
X = df[sem1_features]
y = df['dropout']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

(3539, 30) (885, 30) 0.321 0.321


**Notes:** I split the data into 3,539 training students (80%), which the model learns from, and 885 test students (20%), which it never sees until grading. Both groups keep the same 32.1% dropout rate (`stratify`), so the test is fair. `random_state=42` makes the split repeatable.

In [26]:
def make_model(features, estimator):
    cats = [c for c in features if c in CATEGORICAL]
    nums = [c for c in features if c not in CATEGORICAL]
    prep = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cats),
        ('num', StandardScaler(), nums),
    ])
    return Pipeline([('prep', prep), ('model', estimator)])

def evaluate(name, features, estimator, cutoff=0.5):
    model = make_model(features, estimator)
    model.fit(X_train[features], y_train)
    prob = model.predict_proba(X_test[features])[:, 1]
    pred = (prob >= cutoff).astype(int)
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_test, pred), 3),
        'recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'roc_auc': round(roc_auc_score(y_test, prob), 3),
    }

**Notes:** `make_model` prepares the data and attaches a model in one pipeline. It one-hot encodes category codes and scales number columns so they're comparable. `evaluate` trains a model and scores it on the test set with 4 metrics:
- **Accuracy:** % of all predictions that were right
- **Recall:** % of real dropouts the model caught (most important here)
- **Precision:** % of flagged students who really dropped out
- **ROC AUC:** how well the model ranks risky students above safe ones (0.5 = coin flip, 1.0 = perfect)

In [27]:
results = [
    evaluate('Baseline (always "not dropout")', sem1_features,
             DummyClassifier(strategy='most_frequent')),
    evaluate('Logistic regression', sem1_features,
             LogisticRegression(max_iter=2000, class_weight='balanced')),
    evaluate('Random forest', sem1_features,
             RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                    class_weight='balanced', random_state=42)),
]
pd.DataFrame(results)

,model,accuracy,recall,precision,roc_auc
0,"Baseline (always ""not dropout"")",0.679,0.000,0.000,0.500
1,Logistic regression,0.860,0.806,0.768,0.913
2,Random forest,0.837,0.782,0.730,0.900


**Takeaway:** The baseline, which always guesses "not dropout," gets 68% accuracy but catches zero dropouts (recall 0). That shows why accuracy alone is misleading with unbalanced data.
- **Logistic regression** did best: 86% accuracy, caught 81% of dropouts, 77% of flagged students really dropped out, AUC 0.913.
- **Random forest** was close but slightly worse: recall 0.764, AUC 0.897.

The simpler model won, and it's also easier to explain to advisors.

In [28]:
rows = []
for stage, feats in [('At enrollment', enroll_features), ('After semester 1', sem1_features)]:
    for name, est in [
        ('Logistic regression', LogisticRegression(max_iter=2000, class_weight='balanced')),
        ('Random forest', RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                 class_weight='balanced', random_state=42)),
    ]:
        r = evaluate(name, feats, est)
        r['stage'] = stage
        rows.append(r)
pd.DataFrame(rows)[['stage', 'model', 'accuracy', 'recall', 'precision', 'roc_auc']]

,stage,model,accuracy,recall,precision,roc_auc
0,At enrollment,Logistic regression,0.760,0.718,0.607,0.836
1,At enrollment,Random forest,0.757,0.701,0.605,0.824
2,After semester 1,Logistic regression,0.860,0.806,0.768,0.913
3,After semester 1,Random forest,0.837,0.782,0.730,0.900


**Takeaway:** Even at enrollment, with no grades, logistic regression ranks students fairly well (AUC 0.836) and catches 72% of dropouts. But only 61% of the students it flags actually drop out, so there are many false alarms. After semester 1, AUC rises to 0.913, recall to 81%, and precision to 77%.
- **Trade-off:** acting on day 1 reaches students earliest, but with more false alarms. Waiting one semester is much more accurate, but some students may already be struggling or gone.
- A real college could use both: light-touch support at enrollment, then targeted outreach after semester 1.

In [29]:
best = make_model(sem1_features, LogisticRegression(max_iter=2000, class_weight='balanced'))
train_prob = cross_val_predict(best, X_train[sem1_features], y_train,
                               cv=5, method='predict_proba')[:, 1]

COST_MISS, COST_FALSE_ALARM = 5, 1

cut = []
for c in np.arange(0.1, 0.91, 0.05):
    pred = (train_prob >= c).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    cut.append({'cutoff': round(c, 2), 'caught': tp, 'missed': fn,
                'false_alarms': fp, 'total_cost': fn * COST_MISS + fp * COST_FALSE_ALARM})
cut = pd.DataFrame(cut)
CUTOFF = cut.loc[cut['total_cost'].idxmin(), 'cutoff']
print('Chosen cutoff:', CUTOFF)
cut

Chosen cutoff: 0.35


,cutoff,caught,missed,false_alarms,total_cost
0,0.10,1097,40,1524,1724
1,0.15,1066,71,1173,1528
2,0.20,1044,93,942,1407
3,0.25,1029,108,782,1322
4,0.30,1008,129,657,1302
5,0.35,985,152,540,1300
6,0.40,952,185,449,1374
7,0.45,925,212,380,1440
8,0.50,897,240,313,1513
9,0.55,871,266,263,1593


**Takeaway:** The model gives each student a risk score from 0 to 1, and the cutoff decides who gets flagged. I set the cost of a missed dropout at 5 and a false alarm at 1, because a missed student can lose their degree while a false alarm costs about an advisor hour.
- The lowest total cost was at a **cutoff of 0.35** (cost 1,300), compared with 1,513 at the default of 0.5.
- A lower cutoff catches more dropouts but creates more false alarms. I chose it using training data only (5-fold cross-validation), so the test set stays unseen.

In [30]:
best.fit(X_train[sem1_features], y_train)
test_prob = best.predict_proba(X_test[sem1_features])[:, 1]
test_pred = (test_prob >= CUTOFF).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
print(f'Caught {tp} of {tp+fn} dropouts ({tp/(tp+fn):.0%}), '
      f'missed {fn}, false alarms {fp}, flagged {tp+fp} of {len(y_test)} students')

Caught 249 of 284 dropouts (88%), missed 35, false alarms 133, flagged 382 of 885 students


**Takeaway:** At a cutoff of 0.35, the model caught **249 of 284 dropouts (88%)** in the test set and missed 35. But it flagged 382 of 885 students (43%), including 133 false alarms. That's a lot of outreach for a real advising office. Next, I'll rank students by risk and see how many real dropouts the office could reach if it can only contact the top 10%.

## Model summary

**Best model: logistic regression.** After semester 1, it caught 81% of dropouts (recall 0.806) with a ROC AUC of 0.913, beating the random forest (recall 0.764, AUC 0.897). The baseline shows why accuracy alone misleads: by always guessing "not dropout," it scores 68% accuracy but catches zero dropouts. The simpler model also wins on explainability, which matters when advisors need to understand why a student was flagged.

**Early vs. accurate:** at enrollment, with no grades yet, the model still ranks students fairly well (AUC 0.836, recall 0.718), but only 61% of the students it flags actually drop out. After semester 1, AUC rises to 0.913 and precision to 77%. Semester 1 results are the biggest boost, so the trade-off is acting early with more false alarms vs. waiting one semester for better accuracy.

**Cutoff: 0.35, not 0.5.** I treated a missed dropout as 5 times as costly as a false alarm, and chose the cutoff with the lowest total cost using training data only. On the test set, it caught 249 of 284 dropouts (88%), missed 35, and flagged 382 of 885 students, including 133 false alarms.

**Next question:** flagging 43% of students is too many for a real advising office. In the outreach step, I'll rank students by risk and focus on the top 10% to see how many real dropouts a limited budget can reach.